# Random Forest Churn Pipeline

This notebook trains a Random Forest churn model, evaluates it on a train/test split, reviews the confusion matrix and classification metrics, then retrains on the full dataset and saves the production artifact to `backend/models/churn_random_forest_pipeline.pkl`.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
TRAIN_PATH = ROOT / 'backend' / 'data' / 'train_churn.csv'
MODEL_PATH = ROOT / 'backend' / 'models' / 'churn_random_forest_pipeline.pkl'

IDENTITY_COLUMN = 'customerID'
TARGET_COLUMN = 'Churn'
NUMERIC_COLUMNS = ['Tenure', 'MonthlyCharges', 'TotalCharges']
CATEGORICAL_COLUMNS = [
    'Gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]


In [ ]:
train_df = pd.read_csv(TRAIN_PATH)
train_df.columns = [column.strip() for column in train_df.columns]
train_df['TotalCharges'] = pd.to_numeric(train_df['TotalCharges'], errors='coerce')
train_df['SeniorCitizen'] = pd.to_numeric(train_df['SeniorCitizen'], errors='coerce').fillna(0).astype(int)
train_df.head()


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline(steps=[
                ('imputer', KNNImputer(n_neighbors=2, weights='distance')),
                ('scaler', StandardScaler()),
            ]),
            NUMERIC_COLUMNS,
        ),
        (
            'cat',
            Pipeline(steps=[
                ('imputer', SimpleImputer(strategy='most_frequent')),
                ('encoder', OneHotEncoder(handle_unknown='ignore')),
            ]),
            CATEGORICAL_COLUMNS,
        ),
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1,
    )),
])
pipeline


In [ ]:
X = train_df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS]
y = train_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

metrics = pd.DataFrame([
    {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label='Yes'),
        'recall': recall_score(y_test, y_pred, pos_label='Yes'),
        'f1': f1_score(y_test, y_pred, pos_label='Yes'),
        'avg_predicted_probability': y_proba.mean(),
    }
])
metrics.round(4)


In [ ]:
confusion = confusion_matrix(y_test, y_pred, labels=['No', 'Yes'])
confusion_df = pd.DataFrame(confusion, index=['Actual No', 'Actual Yes'], columns=['Predicted No', 'Predicted Yes'])
confusion_df


In [ ]:
report = pd.DataFrame(classification_report(y_test, y_pred, output_dict=True, zero_division=0)).transpose()
report.round(4)


In [ ]:
production_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(
        n_estimators=250,
        max_depth=12,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=1,
    )),
])
production_pipeline.fit(X, y)

artifact = {
    'pipeline': production_pipeline,
    'sklearn_version': sklearn_version,
    'trained_at': pd.Timestamp.utcnow().isoformat(),
    'metrics': metrics.iloc[0].round(4).to_dict(),
    'confusion_matrix': confusion_df.values.tolist(),
    'classification_report': report.round(4).to_dict(),
    'feature_columns': NUMERIC_COLUMNS + CATEGORICAL_COLUMNS,
    'train_size': int(len(X_train)),
    'test_size': int(len(X_test)),
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(artifact, MODEL_PATH)
MODEL_PATH
